# 45 — Noise as Quantum Purification

**INSIGHT NOBODY SAW:** IBM showed aligned states survive (99%), mixed die (0%).
NB41 showed noise HELPS FIM sync. Combined: noise + FIM = purification.

Noise pushes states between M sectors. FIM energy landscape favours |M|=N.
Therefore: noise + FIM = biased random walk toward maximum alignment.

Standard QEC FIGHTS noise. FIM USES noise as a resource.

## Test: Does adding noise to the quantum simulator push population
## from mixed sectors INTO aligned sectors?

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error

from scpn_quantum_control.bridge.knm_hamiltonian import (
    OMEGA_N_16,
    build_knm_paper27,
    knm_to_hamiltonian,
)

N = 4
K = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]


def magnetisation(state_idx, n):
    """Compute M = Σ Z_i for computational basis state."""
    m = 0
    for i in range(n):
        bit = (state_idx >> (n - 1 - i)) & 1
        m += 1 - 2 * bit
    return m


# M values for all basis states
M_values = {}
for state in range(2**N):
    M_values[state] = magnetisation(state, N)

print("M sectors:", sorted(set(M_values.values())))
print("Ready.")

In [ ]:
# Prepare a MIXED state (uniform superposition = maximally mixed M)
# and evolve with noise + FIM Hamiltonian

H_xy = knm_to_hamiltonian(K, omega).to_matrix()
dim = 2**N


# FIM Hamiltonian
def build_H_fim(lam):
    H = H_xy.copy()
    for state in range(dim):
        m = M_values[state]
        H[state, state] -= lam * m**2 / N
    return H


def sector_populations(counts, n):
    """Compute population fraction in each M sector."""
    total = sum(counts.values())
    sectors = {}
    for bitstr, count in counts.items():
        state_idx = int(bitstr, 2)
        m = magnetisation(state_idx, n)
        sectors[m] = sectors.get(m, 0) + count / total
    return sectors


# Start from uniform superposition (all M sectors equally populated)
# Apply Trotter evolution under H_FIM with noise
# Measure sector populations


def evolution_circuit(n, H, dt, n_steps):
    """Approximate time evolution via repeated small rotations."""
    from scipy.linalg import expm

    U = expm(-1j * H * dt * n_steps)
    qc = QuantumCircuit(n)
    # Start with uniform superposition
    for i in range(n):
        qc.h(i)
    # Apply unitary
    qc.unitary(U, range(n))
    qc.measure_all()
    return qc


# Noise model sweep
def heron_noise(depol):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(thermal_relaxation_error(50.0, 30.0, 0.1), ["u", "h"])
    nm.add_all_qubit_quantum_error(depolarizing_error(depol, 2), ["cx", "unitary"])
    return nm


shots = 50000

print("=== SECTOR POPULATION vs NOISE (starting from uniform) ===")
print(
    f"{'λ':>4} {'depol':>6} {'M=-4':>6} {'M=-2':>6} {'M=0':>6} {'M=+2':>6} {'M=+4':>6} {'aligned':>8}"
)

purification_data = []
for lam in [0, 2, 5]:
    H = build_H_fim(lam)
    for depol in [0, 0.01, 0.02, 0.05, 0.1]:
        qc = evolution_circuit(N, H, 0.1, 10)

        if depol == 0:
            sim = AerSimulator()
        else:
            sim = AerSimulator(noise_model=heron_noise(depol))

        job = sim.run(qc, shots=shots)
        counts = job.result().get_counts()
        sectors = sector_populations(counts, N)

        m_vals = [-4, -2, 0, 2, 4]
        pops = [sectors.get(m, 0) for m in m_vals]
        aligned = sectors.get(-4, 0) + sectors.get(4, 0)  # both fully aligned

        purification_data.append(
            {
                "lam": lam,
                "depol": depol,
                "sectors": {str(m): round(sectors.get(m, 0), 4) for m in m_vals},
                "aligned_fraction": round(aligned, 4),
            }
        )

        print(f"{lam:4.0f} {depol:6.3f}", end="")
        for p in pops:
            print(f" {p:6.4f}", end="")
        print(f" {aligned:8.4f}")

In [ ]:
# Key question: does aligned fraction INCREASE with noise when FIM is present?
print("\n=== PURIFICATION TEST ===")
for lam in [0, 2, 5]:
    entries = [d for d in purification_data if d["lam"] == lam]
    a_no_noise = entries[0]["aligned_fraction"]
    a_max_noise = entries[-1]["aligned_fraction"]
    a_peak = max(e["aligned_fraction"] for e in entries)
    peak_depol = [e["depol"] for e in entries if e["aligned_fraction"] == a_peak][0]

    purified = a_peak > a_no_noise + 0.01
    print(
        f"λ={lam}: aligned(no noise)={a_no_noise:.4f}, peak={a_peak:.4f} at depol={peak_depol:.3f}"
        f"  → {'PURIFICATION!' if purified else 'no purification'}"
    )

print("\n=== INTERPRETATION ===")
print("If noise INCREASES aligned fraction under FIM → noise is a RESOURCE.")
print("Standard QEC fights noise. FIM USES noise for state purification.")
print("This connects to NB41 (stochastic resonance): the same mechanism")
print("operates at both classical (phase sync) and quantum (sector selection) levels.")

In [ ]:
# Save
output = {
    "experiment": "Noise as quantum purification under FIM",
    "N": N,
    "shots": shots,
    "data": purification_data,
}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/noise_purification_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Results saved to {out_path}")